# AI Stock Market Predictor — Google Colab Notebook

This notebook converts your uploaded `stock_predictor.zip` project into a Google Colab runnable file. It includes Flask dashboard, Yahoo Finance stock data, quick prediction, LSTM model training, moving averages, and sentiment analysis.

**How to use:** Run cells from top to bottom. After the Flask cell runs, Colab will open the dashboard window.

## 1. Install Required Libraries

This uses flexible versions because Colab's Python/TensorFlow versions change over time.

In [1]:
!pip -q install yfinance textblob flask plotly scikit-learn pandas numpy beautifulsoup4 requests
# TensorFlow is usually pre-installed in Colab. Install only if needed.
try:
    import tensorflow as tf
    print('✅ TensorFlow available:', tf.__version__)
except Exception:
    print('Installing TensorFlow...')
    !pip -q install tensorflow


✅ TensorFlow available: 2.20.0


## 2. Create Project Files in Colab

This cell writes your original project files into `/content/stock_predictor`.

In [2]:

from pathlib import Path
import os

base = Path("/content/stock_predictor")
(base / "templates").mkdir(parents=True, exist_ok=True)
(base / "models").mkdir(parents=True, exist_ok=True)

files = {
    "requirements.txt": 'flask==2.3.3\nyfinance==0.2.31\npandas==2.0.3\nnumpy==1.24.3\nscikit-learn==1.3.0\ntensorflow==2.12.0\nplotly==5.17.0\ntextblob==0.17.1\nrequests==2.31.0\nbeautifulsoup4==4.12.2\n',
    "README.md": '# 📈 AI Stock Market Predictor\n\nFinal Year Project — LSTM Neural Network + Sentiment Analysis + Real-time Dashboard\n\n---\n\n## 🚀 Setup — Step by Step\n\n### Step 1: Environment banao\n```bash\nconda create -n stock_predictor python=3.9\nconda activate stock_predictor\n```\n\n### Step 2: Libraries install karo\n```bash\npip install -r requirements.txt\n```\n\n### Step 3: Project chalao\n```bash\npython app.py\n```\n\n### Step 4: Browser mein kholo\n```\nhttp://127.0.0.1:5000\n```\n\n---\n\n## 📁 Project Structure\n\n```\nstock_predictor/\n├── app.py              ← Flask server (main file)\n├── lstm_model.py       ← LSTM training code\n├── sentiment.py        ← News sentiment analysis\n├── requirements.txt    ← All libraries\n├── run.bat             ← Windows me double click karo\n├── templates/\n│   └── index.html      ← Dashboard UI\n└── models/             ← Trained models save honge yahan\n```\n\n---\n\n## 🎯 Features\n\n- ✅ Real-time stock data (Yahoo Finance)\n- ✅ LSTM model se 30-day price prediction\n- ✅ News sentiment analysis\n- ✅ Moving averages (MA20, MA50)\n- ✅ Volume analysis\n- ✅ Interactive Plotly charts\n- ✅ 5 quick stocks (AAPL, GOOGL, MSFT, TSLA, AMZN)\n\n---\n\n## 💡 How to Use\n\n1. Ticker enter karo (e.g. AAPL)\n2. **Analyze** — historical data + sentiment dekho\n3. **Quick Predict** — simple trend prediction\n4. **Train LSTM** — accurate ML prediction (2-3 min)\n\n---\n\n## 🛠️ Technologies\n\n| Technology | Use |\n|------------|-----|\n| Python 3.9 | Core language |\n| Flask | Web server |\n| TensorFlow/Keras | LSTM model |\n| yfinance | Real stock data |\n| Plotly | Interactive charts |\n| TextBlob | Sentiment analysis |\n| Pandas/NumPy | Data processing |\n',
    "app.py": 'from flask import Flask, render_template, request, jsonify\nimport yfinance as yf\nimport pandas as pd\nimport numpy as np\nimport json\nimport os\nimport pickle\nfrom datetime import datetime, timedelta\nfrom sentiment import get_stock_news_sentiment\n\napp = Flask(__name__)\n\n# ══════════════════════════════════════════\n# HELPER FUNCTIONS\n# ══════════════════════════════════════════\n\ndef get_stock_data(ticker, period="6mo"):\n    """Stock ka real data fetch karo"""\n    try:\n        stock = yf.Ticker(ticker)\n        df    = stock.history(period=period)\n        info  = stock.info\n        return df, info\n    except Exception as e:\n        return None, {}\n\ndef get_predictions(ticker, days=30):\n    """Saved model se predict karo"""\n    model_path  = f"models/{ticker}_lstm.h5"\n    scaler_path = f"models/{ticker}_scaler.pkl"\n\n    if not os.path.exists(model_path):\n        return None, "Model nahi mila — pehle train karo"\n\n    try:\n        from tensorflow.keras.models import load_model\n        model = load_model(model_path)\n\n        with open(scaler_path, "rb") as f:\n            scaler = pickle.load(f)\n\n        df     = yf.download(ticker, period="6mo", progress=False)\n        scaled = scaler.transform(df[[\'Close\']])\n        last60 = scaled[-60:].reshape(1, 60, 1)\n\n        preds = []\n        batch = last60.copy()\n        for _ in range(days):\n            p = model.predict(batch, verbose=0)[0][0]\n            preds.append(p)\n            batch = np.append(batch[:, 1:, :], [[[p]]], axis=1)\n\n        preds = scaler.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()\n\n        # Future dates banao\n        last_date = df.index[-1]\n        dates = []\n        for i in range(1, days + 1):\n            d = last_date + timedelta(days=i)\n            while d.weekday() >= 5:\n                d += timedelta(days=1)\n            dates.append(d.strftime("%Y-%m-%d"))\n\n        return {"dates": dates, "prices": preds.tolist()}, None\n\n    except Exception as e:\n        return None, str(e)\n\n# ══════════════════════════════════════════\n# ROUTES\n# ══════════════════════════════════════════\n\n@app.route("/")\ndef home():\n    return render_template("index.html")\n\n@app.route("/analyze", methods=["POST"])\ndef analyze():\n    """Main analysis endpoint"""\n    data   = request.json\n    ticker = data.get("ticker", "AAPL").upper().strip()\n    period = data.get("period", "6mo")\n\n    # Stock data\n    df, info = get_stock_data(ticker, period)\n    if df is None or df.empty:\n        return jsonify({"error": f"\'{ticker}\' ka data nahi mila!"})\n\n    # Current price info\n    current_price  = round(df[\'Close\'].iloc[-1], 2)\n    prev_price     = round(df[\'Close\'].iloc[-2], 2)\n    price_change   = round(current_price - prev_price, 2)\n    price_change_p = round((price_change / prev_price) * 100, 2)\n\n    # Historical chart data\n    chart_data = {\n        "dates":  df.index.strftime("%Y-%m-%d").tolist(),\n        "close":  df[\'Close\'].round(2).tolist(),\n        "open":   df[\'Open\'].round(2).tolist(),\n        "high":   df[\'High\'].round(2).tolist(),\n        "low":    df[\'Low\'].round(2).tolist(),\n        "volume": df[\'Volume\'].tolist()\n    }\n\n    # Moving averages\n    df[\'MA20\'] = df[\'Close\'].rolling(20).mean()\n    df[\'MA50\'] = df[\'Close\'].rolling(50).mean()\n    chart_data[\'ma20\'] = df[\'MA20\'].round(2).fillna(0).tolist()\n    chart_data[\'ma50\'] = df[\'MA50\'].round(2).fillna(0).tolist()\n\n    # Predictions\n    preds, pred_error = get_predictions(ticker)\n\n    # Sentiment\n    sentiment = get_stock_news_sentiment(ticker)\n\n    # Stats\n    stats = {\n        "current_price":    current_price,\n        "price_change":     price_change,\n        "price_change_p":   price_change_p,\n        "high_52w":         round(df[\'High\'].max(), 2),\n        "low_52w":          round(df[\'Low\'].min(), 2),\n        "avg_volume":       int(df[\'Volume\'].mean()),\n        "company_name":     info.get(\'longName\', ticker),\n        "sector":           info.get(\'sector\', \'N/A\'),\n        "market_cap":       info.get(\'marketCap\', 0),\n    }\n\n    return jsonify({\n        "ticker":      ticker,\n        "stats":       stats,\n        "chart_data":  chart_data,\n        "predictions": preds,\n        "pred_error":  pred_error,\n        "sentiment":   sentiment\n    })\n\n@app.route("/train", methods=["POST"])\ndef train():\n    """Model train karo"""\n    data   = request.json\n    ticker = data.get("ticker", "AAPL").upper()\n\n    try:\n        from lstm_model import train_and_save\n        train_and_save(ticker)\n        return jsonify({"success": True, "message": f"✅ {ticker} ka model train ho gaya!"})\n    except Exception as e:\n        return jsonify({"success": False, "message": str(e)})\n\n@app.route("/quick_predict", methods=["POST"])\ndef quick_predict():\n    """Bina LSTM ke simple prediction"""\n    data   = request.json\n    ticker = data.get("ticker", "AAPL").upper()\n\n    try:\n        df = yf.download(ticker, period="3mo", progress=False)\n        closes = df[\'Close\'].values\n\n        # Simple moving average prediction\n        ma7  = closes[-7:].mean()\n        ma14 = closes[-14:].mean()\n        trend = (ma7 - ma14) / ma14 * 100\n\n        last_price = closes[-1]\n        predictions = []\n        dates = []\n        last_date = df.index[-1]\n\n        for i in range(1, 31):\n            factor = 1 + (trend / 100) * (i / 30)\n            predictions.append(round(last_price * factor, 2))\n            d = last_date + timedelta(days=i)\n            while d.weekday() >= 5:\n                d += timedelta(days=1)\n            dates.append(d.strftime("%Y-%m-%d"))\n\n        return jsonify({\n            "dates":  dates,\n            "prices": predictions,\n            "trend":  round(trend, 2)\n        })\n    except Exception as e:\n        return jsonify({"error": str(e)})\n\nif __name__ == "__main__":\n    os.makedirs("models", exist_ok=True)\n    print("🚀 Stock Market Predictor chal raha hai!")\n    print("📊 Browser mein kholo: http://127.0.0.1:5000")\n    app.run(debug=True)\n',
    "lstm_model.py": 'import numpy as np\nimport pandas as pd\nimport yfinance as yf\nfrom sklearn.preprocessing import MinMaxScaler\nfrom tensorflow.keras.models import Sequential\nfrom tensorflow.keras.layers import LSTM, Dense, Dropout\nfrom tensorflow.keras.callbacks import EarlyStopping\nimport pickle\nimport os\n\n# ══════════════════════════════════════════\n# LSTM MODEL — Stock Price Predictor\n# ══════════════════════════════════════════\n\ndef fetch_data(ticker="AAPL", period="2y"):\n    """Yahoo Finance se real data fetch karo"""\n    print(f"📈 {ticker} ka data download ho raha hai...")\n    df = yf.download(ticker, period=period, progress=False)\n    df = df[[\'Close\']].dropna()\n    print(f"✅ {len(df)} days ka data mila!")\n    return df\n\ndef prepare_data(df, look_back=60):\n    """LSTM ke liye data tayyar karo"""\n    scaler = MinMaxScaler(feature_range=(0, 1))\n    scaled = scaler.fit_transform(df[[\'Close\']])\n\n    X, y = [], []\n    for i in range(look_back, len(scaled)):\n        X.append(scaled[i - look_back:i, 0])\n        y.append(scaled[i, 0])\n\n    X, y = np.array(X), np.array(y)\n    X = X.reshape((X.shape[0], X.shape[1], 1))\n\n    # 80% train, 20% test\n    split = int(len(X) * 0.8)\n    return X[:split], X[split:], y[:split], y[split:], scaler\n\ndef build_model(look_back=60):\n    """LSTM model banao"""\n    model = Sequential([\n        LSTM(50, return_sequences=True, input_shape=(look_back, 1)),\n        Dropout(0.2),\n        LSTM(50, return_sequences=False),\n        Dropout(0.2),\n        Dense(25),\n        Dense(1)\n    ])\n    model.compile(optimizer=\'adam\', loss=\'mean_squared_error\')\n    return model\n\ndef train_and_save(ticker="AAPL"):\n    """Model train karo aur save karo"""\n    # Data fetch\n    df = fetch_data(ticker)\n\n    # Prepare\n    X_train, X_test, y_train, y_test, scaler = prepare_data(df)\n\n    # Model banao\n    print("🧠 LSTM model train ho raha hai...")\n    model = build_model()\n\n    early_stop = EarlyStopping(monitor=\'val_loss\', patience=5, restore_best_weights=True)\n\n    model.fit(\n        X_train, y_train,\n        epochs=30,\n        batch_size=32,\n        validation_split=0.1,\n        callbacks=[early_stop],\n        verbose=1\n    )\n\n    # Save karo\n    os.makedirs("models", exist_ok=True)\n    model.save(f"models/{ticker}_lstm.h5")\n    with open(f"models/{ticker}_scaler.pkl", "wb") as f:\n        pickle.dump(scaler, f)\n\n    print(f"✅ Model saved: models/{ticker}_lstm.h5")\n    return model, scaler, df\n\ndef predict_future(ticker="AAPL", days=30):\n    """Future prices predict karo"""\n    from tensorflow.keras.models import load_model\n\n    model  = load_model(f"models/{ticker}_lstm.h5")\n    with open(f"models/{ticker}_scaler.pkl", "rb") as f:\n        scaler = pickle.load(f)\n\n    df     = fetch_data(ticker, period="6mo")\n    scaled = scaler.transform(df[[\'Close\']])\n\n    # Last 60 days se predict karo\n    last_60 = scaled[-60:]\n    predictions = []\n\n    current_batch = last_60.reshape(1, 60, 1)\n    for _ in range(days):\n        pred = model.predict(current_batch, verbose=0)[0][0]\n        predictions.append(pred)\n        current_batch = np.append(current_batch[:, 1:, :], [[[pred]]], axis=1)\n\n    # Wapas original scale par\n    predictions = scaler.inverse_transform(np.array(predictions).reshape(-1, 1))\n    return predictions.flatten().tolist()\n\nif __name__ == "__main__":\n    train_and_save("AAPL")\n    print("\\n🎉 Training complete!")\n',
    "sentiment.py": 'from textblob import TextBlob\nimport yfinance as yf\n\n# ══════════════════════════════════════════\n# SENTIMENT ANALYSIS\n# ══════════════════════════════════════════\n\ndef analyze_sentiment(text):\n    """TextBlob se sentiment nikalo"""\n    blob = TextBlob(str(text))\n    polarity = blob.sentiment.polarity  # -1 (negative) to +1 (positive)\n\n    if polarity > 0.1:\n        label = "Positive 📈"\n        color = "green"\n    elif polarity < -0.1:\n        label = "Negative 📉"\n        color = "red"\n    else:\n        label = "Neutral ➡️"\n        color = "gray"\n\n    return {\n        "polarity":   round(polarity, 3),\n        "label":      label,\n        "color":      color,\n        "confidence": round(abs(polarity) * 100, 1)\n    }\n\ndef get_stock_news_sentiment(ticker="AAPL"):\n    """Yahoo Finance news se sentiment nikalo"""\n    try:\n        stock = yf.Ticker(ticker)\n        news  = stock.news[:10]  # Last 10 news\n\n        results = []\n        total_polarity = 0\n\n        for item in news:\n            title     = item.get(\'title\', \'\')\n            sentiment = analyze_sentiment(title)\n            total_polarity += sentiment[\'polarity\']\n\n            results.append({\n                "title":     title,\n                "sentiment": sentiment[\'label\'],\n                "polarity":  sentiment[\'polarity\'],\n                "color":     sentiment[\'color\'],\n                "link":      item.get(\'link\', \'#\')\n            })\n\n        # Overall sentiment\n        avg = total_polarity / len(results) if results else 0\n        overall = analyze_sentiment(avg)\n\n        return {\n            "news":    results,\n            "overall": overall,\n            "count":   len(results)\n        }\n\n    except Exception as e:\n        return {\n            "news":    [],\n            "overall": {"label": "No data", "polarity": 0, "color": "gray"},\n            "count":   0,\n            "error":   str(e)\n        }\n\nif __name__ == "__main__":\n    result = get_stock_news_sentiment("AAPL")\n    print(f"Overall: {result[\'overall\'][\'label\']}")\n    for n in result[\'news\'][:3]:\n        print(f"  {n[\'sentiment\']} — {n[\'title\'][:60]}...")\n',
    "templates/index.html": '<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta charset="UTF-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1.0">\n  <title>AI Stock Market Predictor</title>\n  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>\n  <style>\n    * { margin: 0; padding: 0; box-sizing: border-box; }\n\n    body {\n      font-family: \'Segoe UI\', sans-serif;\n      background: #0d1117;\n      color: #e6edf3;\n      min-height: 100vh;\n    }\n\n    /* ── HEADER ── */\n    .header {\n      background: linear-gradient(135deg, #161b22, #21262d);\n      border-bottom: 1px solid #30363d;\n      padding: 16px 24px;\n      display: flex;\n      align-items: center;\n      justify-content: space-between;\n    }\n    .logo { font-size: 20px; font-weight: 700; color: #58a6ff; }\n    .logo span { color: #3fb950; }\n    .subtitle { font-size: 12px; color: #8b949e; margin-top: 2px; }\n\n    /* ── SEARCH BAR ── */\n    .search-section {\n      background: #161b22;\n      border-bottom: 1px solid #30363d;\n      padding: 16px 24px;\n      display: flex;\n      gap: 10px;\n      align-items: center;\n      flex-wrap: wrap;\n    }\n    .search-input {\n      background: #21262d;\n      border: 1px solid #30363d;\n      border-radius: 8px;\n      padding: 10px 16px;\n      color: #e6edf3;\n      font-size: 15px;\n      width: 200px;\n      text-transform: uppercase;\n      outline: none;\n    }\n    .search-input:focus { border-color: #58a6ff; }\n    .period-select {\n      background: #21262d;\n      border: 1px solid #30363d;\n      border-radius: 8px;\n      padding: 10px 12px;\n      color: #e6edf3;\n      font-size: 13px;\n      outline: none;\n    }\n    .btn {\n      padding: 10px 20px;\n      border-radius: 8px;\n      border: none;\n      font-size: 13px;\n      font-weight: 600;\n      cursor: pointer;\n      transition: opacity 0.2s;\n    }\n    .btn:hover { opacity: 0.85; }\n    .btn-primary { background: #238636; color: white; }\n    .btn-blue    { background: #1f6feb; color: white; }\n    .btn-orange  { background: #bd561d; color: white; }\n    .btn-sm { padding: 6px 14px; font-size: 12px; }\n\n    /* ── MAIN LAYOUT ── */\n    .main { padding: 20px 24px; max-width: 1400px; margin: 0 auto; }\n\n    /* ── STATS CARDS ── */\n    .stats-row {\n      display: grid;\n      grid-template-columns: repeat(auto-fit, minmax(160px, 1fr));\n      gap: 12px;\n      margin-bottom: 20px;\n    }\n    .stat-card {\n      background: #161b22;\n      border: 1px solid #30363d;\n      border-radius: 10px;\n      padding: 14px 16px;\n    }\n    .stat-label { font-size: 11px; color: #8b949e; margin-bottom: 4px; text-transform: uppercase; letter-spacing: 0.05em; }\n    .stat-value { font-size: 22px; font-weight: 700; color: #e6edf3; }\n    .stat-change { font-size: 12px; margin-top: 3px; }\n    .green { color: #3fb950; }\n    .red   { color: #f85149; }\n    .gray  { color: #8b949e; }\n\n    /* ── CHARTS ── */\n    .charts-grid {\n      display: grid;\n      grid-template-columns: 2fr 1fr;\n      gap: 16px;\n      margin-bottom: 16px;\n    }\n    .chart-card {\n      background: #161b22;\n      border: 1px solid #30363d;\n      border-radius: 10px;\n      padding: 16px;\n    }\n    .chart-title {\n      font-size: 14px;\n      font-weight: 600;\n      color: #e6edf3;\n      margin-bottom: 12px;\n      display: flex;\n      align-items: center;\n      justify-content: space-between;\n    }\n    .chart-title span { font-size: 11px; color: #8b949e; font-weight: 400; }\n\n    /* ── BOTTOM GRID ── */\n    .bottom-grid {\n      display: grid;\n      grid-template-columns: 1fr 1fr;\n      gap: 16px;\n    }\n\n    /* ── NEWS SENTIMENT ── */\n    .news-item {\n      border-bottom: 1px solid #21262d;\n      padding: 10px 0;\n      display: flex;\n      align-items: flex-start;\n      gap: 10px;\n    }\n    .news-item:last-child { border-bottom: none; }\n    .news-dot { width: 8px; height: 8px; border-radius: 50%; flex-shrink: 0; margin-top: 5px; }\n    .news-title { font-size: 12px; color: #c9d1d9; line-height: 1.5; }\n    .news-sent  { font-size: 10px; margin-top: 3px; }\n\n    /* ── PREDICTION SECTION ── */\n    .predict-controls {\n      display: flex;\n      gap: 8px;\n      margin-bottom: 12px;\n      flex-wrap: wrap;\n    }\n\n    /* ── LOADING ── */\n    .loading {\n      display: none;\n      text-align: center;\n      padding: 40px;\n      color: #8b949e;\n    }\n    .spinner {\n      width: 32px; height: 32px;\n      border: 3px solid #30363d;\n      border-top-color: #58a6ff;\n      border-radius: 50%;\n      animation: spin 0.8s linear infinite;\n      margin: 0 auto 12px;\n    }\n    @keyframes spin { to { transform: rotate(360deg); } }\n\n    /* ── ALERT ── */\n    .alert {\n      padding: 12px 16px;\n      border-radius: 8px;\n      font-size: 13px;\n      margin-bottom: 14px;\n      display: none;\n    }\n    .alert-error { background: #2d1b1b; border: 1px solid #f85149; color: #f85149; }\n    .alert-success { background: #1b2d1b; border: 1px solid #3fb950; color: #3fb950; }\n\n    /* ── QUICK STOCKS ── */\n    .quick-stocks { display: flex; gap: 8px; flex-wrap: wrap; align-items: center; }\n    .quick-btn {\n      background: #21262d;\n      border: 1px solid #30363d;\n      border-radius: 6px;\n      padding: 5px 12px;\n      font-size: 12px;\n      color: #8b949e;\n      cursor: pointer;\n    }\n    .quick-btn:hover { background: #30363d; color: #e6edf3; }\n\n    .overall-sentiment {\n      background: #21262d;\n      border-radius: 8px;\n      padding: 12px 14px;\n      margin-bottom: 12px;\n      display: flex;\n      align-items: center;\n      gap: 10px;\n    }\n    .sent-label { font-size: 13px; font-weight: 600; }\n    .sent-desc  { font-size: 11px; color: #8b949e; }\n\n    @media (max-width: 768px) {\n      .charts-grid, .bottom-grid { grid-template-columns: 1fr; }\n      .search-input { width: 140px; }\n    }\n  </style>\n</head>\n<body>\n\n<!-- HEADER -->\n<div class="header">\n  <div>\n    <div class="logo">📈 AI Stock <span>Predictor</span></div>\n    <div class="subtitle">LSTM Neural Network + Sentiment Analysis + Real-time Data</div>\n  </div>\n  <div style="font-size:12px; color:#8b949e;">Final Year Project</div>\n</div>\n\n<!-- SEARCH -->\n<div class="search-section">\n  <input type="text" id="tickerInput" class="search-input" placeholder="AAPL, GOOGL..." value="AAPL">\n  <select id="periodSelect" class="period-select">\n    <option value="1mo">1 Month</option>\n    <option value="3mo">3 Months</option>\n    <option value="6mo" selected>6 Months</option>\n    <option value="1y">1 Year</option>\n    <option value="2y">2 Years</option>\n  </select>\n  <button class="btn btn-primary" onclick="analyzeStock()">🔍 Analyze</button>\n  <button class="btn btn-blue" onclick="quickPredict()">⚡ Quick Predict</button>\n  <button class="btn btn-orange" onclick="trainModel()">🧠 Train LSTM</button>\n\n  <div class="quick-stocks">\n    <span style="font-size:11px; color:#8b949e;">Quick:</span>\n    <button class="quick-btn" onclick="setTicker(\'AAPL\')">AAPL</button>\n    <button class="quick-btn" onclick="setTicker(\'GOOGL\')">GOOGL</button>\n    <button class="quick-btn" onclick="setTicker(\'MSFT\')">MSFT</button>\n    <button class="quick-btn" onclick="setTicker(\'TSLA\')">TSLA</button>\n    <button class="quick-btn" onclick="setTicker(\'AMZN\')">AMZN</button>\n  </div>\n</div>\n\n<div class="main">\n  <div id="alertBox" class="alert"></div>\n  <div id="loading" class="loading">\n    <div class="spinner"></div>\n    <div>Data fetch ho raha hai...</div>\n  </div>\n\n  <div id="dashboard" style="display:none;">\n\n    <!-- STATS -->\n    <div class="stats-row" id="statsRow"></div>\n\n    <!-- MAIN CHARTS -->\n    <div class="charts-grid">\n      <div class="chart-card">\n        <div class="chart-title">\n          Price History <span id="companyName"></span>\n        </div>\n        <div id="priceChart" style="height:300px;"></div>\n      </div>\n      <div class="chart-card">\n        <div class="chart-title">Volume</div>\n        <div id="volumeChart" style="height:300px;"></div>\n      </div>\n    </div>\n\n    <!-- PREDICTION + SENTIMENT -->\n    <div class="bottom-grid">\n      <div class="chart-card">\n        <div class="chart-title">\n          🔮 Price Prediction (30 days)\n          <span id="predStatus" style="color:#f0a500;"></span>\n        </div>\n        <div id="predChart" style="height:280px;"></div>\n      </div>\n      <div class="chart-card">\n        <div class="chart-title">📰 News Sentiment Analysis</div>\n        <div id="overallSentiment" class="overall-sentiment" style="display:none;"></div>\n        <div id="newsList" style="max-height:280px; overflow-y:auto;"></div>\n      </div>\n    </div>\n\n  </div>\n</div>\n\n<script>\n  let currentData = null;\n\n  function setTicker(t) {\n    document.getElementById(\'tickerInput\').value = t;\n    analyzeStock();\n  }\n\n  function showAlert(msg, type=\'error\') {\n    const box = document.getElementById(\'alertBox\');\n    box.className = `alert alert-${type}`;\n    box.textContent = msg;\n    box.style.display = \'block\';\n    setTimeout(() => box.style.display = \'none\', 5000);\n  }\n\n  async function analyzeStock() {\n    const ticker = document.getElementById(\'tickerInput\').value.trim().toUpperCase();\n    const period = document.getElementById(\'periodSelect\').value;\n    if (!ticker) { showAlert(\'Ticker enter karo!\'); return; }\n\n    document.getElementById(\'loading\').style.display = \'block\';\n    document.getElementById(\'dashboard\').style.display = \'none\';\n\n    try {\n      const res  = await fetch(\'/analyze\', {\n        method: \'POST\',\n        headers: {\'Content-Type\':\'application/json\'},\n        body: JSON.stringify({ticker, period})\n      });\n      const data = await res.json();\n\n      if (data.error) { showAlert(data.error); return; }\n\n      currentData = data;\n      renderDashboard(data);\n    } catch(e) {\n      showAlert(\'Server error: \' + e.message);\n    } finally {\n      document.getElementById(\'loading\').style.display = \'none\';\n    }\n  }\n\n  function renderDashboard(data) {\n    const { stats, chart_data, predictions, sentiment } = data;\n    document.getElementById(\'dashboard\').style.display = \'block\';\n\n    // Company name\n    document.getElementById(\'companyName\').textContent = stats.company_name || data.ticker;\n\n    // Stats cards\n    const changeClass = stats.price_change >= 0 ? \'green\' : \'red\';\n    const changeArrow = stats.price_change >= 0 ? \'▲\' : \'▼\';\n    document.getElementById(\'statsRow\').innerHTML = `\n      <div class="stat-card">\n        <div class="stat-label">Current Price</div>\n        <div class="stat-value">$${stats.current_price}</div>\n        <div class="stat-change ${changeClass}">${changeArrow} $${Math.abs(stats.price_change)} (${Math.abs(stats.price_change_p)}%)</div>\n      </div>\n      <div class="stat-card">\n        <div class="stat-label">52W High</div>\n        <div class="stat-value green">$${stats.high_52w}</div>\n      </div>\n      <div class="stat-card">\n        <div class="stat-label">52W Low</div>\n        <div class="stat-value red">$${stats.low_52w}</div>\n      </div>\n      <div class="stat-card">\n        <div class="stat-label">Avg Volume</div>\n        <div class="stat-value" style="font-size:16px;">${(stats.avg_volume/1e6).toFixed(1)}M</div>\n      </div>\n      <div class="stat-card">\n        <div class="stat-label">Sector</div>\n        <div class="stat-value" style="font-size:14px;">${stats.sector}</div>\n      </div>\n      <div class="stat-card">\n        <div class="stat-label">Market Cap</div>\n        <div class="stat-value" style="font-size:15px;">${stats.market_cap ? \'$\'+(stats.market_cap/1e9).toFixed(0)+\'B\' : \'N/A\'}</div>\n      </div>\n    `;\n\n    // Price chart with MA\n    const priceTraces = [\n      { x: chart_data.dates, y: chart_data.close, type:\'scatter\', name:\'Close\', line:{color:\'#58a6ff\', width:2} },\n      { x: chart_data.dates, y: chart_data.ma20,  type:\'scatter\', name:\'MA20\',  line:{color:\'#f0a500\', width:1, dash:\'dot\'} },\n      { x: chart_data.dates, y: chart_data.ma50,  type:\'scatter\', name:\'MA50\',  line:{color:\'#ff7b72\', width:1, dash:\'dot\'} },\n    ];\n    Plotly.newPlot(\'priceChart\', priceTraces, darkLayout(\'\'), {responsive:true, displayModeBar:false});\n\n    // Volume chart\n    Plotly.newPlot(\'volumeChart\', [{\n      x: chart_data.dates, y: chart_data.volume, type:\'bar\',\n      marker:{color: chart_data.close.map((c,i) => i>0 && c >= chart_data.close[i-1] ? \'#3fb950\' : \'#f85149\')}\n    }], darkLayout(\'\'), {responsive:true, displayModeBar:false});\n\n    // Prediction chart\n    if (predictions && predictions.dates) {\n      document.getElementById(\'predStatus\').textContent = \'(LSTM Model)\';\n      Plotly.newPlot(\'predChart\', [\n        { x: chart_data.dates.slice(-30), y: chart_data.close.slice(-30), name:\'Actual\', line:{color:\'#58a6ff\'} },\n        { x: predictions.dates, y: predictions.prices, name:\'Predicted\', line:{color:\'#3fb950\', dash:\'dash\'}, fill:\'tonexty\', fillcolor:\'rgba(63,185,80,0.1)\' }\n      ], darkLayout(\'\'), {responsive:true, displayModeBar:false});\n    } else {\n      document.getElementById(\'predStatus\').textContent = \'(Model train karo)\';\n      document.getElementById(\'predChart\').innerHTML = `\n        <div style="text-align:center; padding:60px 20px; color:#8b949e;">\n          <div style="font-size:32px; margin-bottom:12px;">🧠</div>\n          <div>"Train LSTM" button dabao accurate prediction ke liye</div>\n          <div style="font-size:11px; margin-top:6px;">Ya "Quick Predict" se simple prediction dekho</div>\n        </div>`;\n    }\n\n    // Sentiment\n    if (sentiment && sentiment.news && sentiment.news.length > 0) {\n      const ov = sentiment.overall;\n      const sentColor = ov.color === \'green\' ? \'#3fb950\' : ov.color === \'red\' ? \'#f85149\' : \'#8b949e\';\n      document.getElementById(\'overallSentiment\').style.display = \'flex\';\n      document.getElementById(\'overallSentiment\').innerHTML = `\n        <div style="width:36px;height:36px;border-radius:50%;background:${sentColor}22;border:2px solid ${sentColor};display:flex;align-items:center;justify-content:center;font-size:18px;flex-shrink:0;">\n          ${ov.color===\'green\'?\'📈\':ov.color===\'red\'?\'📉\':\'➡️\'}\n        </div>\n        <div>\n          <div class="sent-label" style="color:${sentColor}">${ov.label}</div>\n          <div class="sent-desc">Based on ${sentiment.count} recent news articles</div>\n        </div>`;\n\n      document.getElementById(\'newsList\').innerHTML = sentiment.news.map(n => `\n        <div class="news-item">\n          <div class="news-dot" style="background:${n.color===\'green\'?\'#3fb950\':n.color===\'red\'?\'#f85149\':\'#8b949e\'}"></div>\n          <div>\n            <div class="news-title"><a href="${n.link}" target="_blank" style="color:#c9d1d9;text-decoration:none;">${n.title}</a></div>\n            <div class="news-sent" style="color:${n.color===\'green\'?\'#3fb950\':n.color===\'red\'?\'#f85149\':\'#8b949e\'}">${n.sentiment} (${n.polarity})</div>\n          </div>\n        </div>`).join(\'\');\n    } else {\n      document.getElementById(\'newsList\').innerHTML = `<div style="padding:20px;text-align:center;color:#8b949e;font-size:13px;">News data available nahi hai</div>`;\n    }\n  }\n\n  async function quickPredict() {\n    const ticker = document.getElementById(\'tickerInput\').value.trim().toUpperCase();\n    if (!ticker) { showAlert(\'Ticker enter karo!\'); return; }\n\n    document.getElementById(\'predStatus\').textContent = \'(Calculating...)\';\n    const res  = await fetch(\'/quick_predict\', {\n      method:\'POST\', headers:{\'Content-Type\':\'application/json\'},\n      body: JSON.stringify({ticker})\n    });\n    const data = await res.json();\n\n    if (data.error) { showAlert(data.error); return; }\n\n    document.getElementById(\'predStatus\').textContent = `(Trend: ${data.trend > 0 ? \'+\' : \'\'}${data.trend}%)`;\n    document.getElementById(\'dashboard\').style.display = \'block\';\n\n    if (currentData) {\n      const cd = currentData.chart_data;\n      Plotly.newPlot(\'predChart\', [\n        { x: cd.dates.slice(-30), y: cd.close.slice(-30), name:\'Actual\',    line:{color:\'#58a6ff\'} },\n        { x: data.dates,          y: data.prices,         name:\'Predicted\', line:{color:\'#f0a500\', dash:\'dash\'} }\n      ], darkLayout(\'\'), {responsive:true, displayModeBar:false});\n    }\n  }\n\n  async function trainModel() {\n    const ticker = document.getElementById(\'tickerInput\').value.trim().toUpperCase();\n    if (!ticker) { showAlert(\'Ticker enter karo!\'); return; }\n\n    showAlert(`🧠 ${ticker} ka LSTM model train ho raha hai — 2-3 minute lagenge...`, \'success\');\n    document.getElementById(\'alertBox\').style.display = \'block\';\n\n    const res  = await fetch(\'/train\', {\n      method:\'POST\', headers:{\'Content-Type\':\'application/json\'},\n      body: JSON.stringify({ticker})\n    });\n    const data = await res.json();\n    showAlert(data.message, data.success ? \'success\' : \'error\');\n    if (data.success) analyzeStock();\n  }\n\n  function darkLayout(title) {\n    return {\n      title: title,\n      paper_bgcolor: \'transparent\',\n      plot_bgcolor:  \'transparent\',\n      font:  { color: \'#8b949e\', size: 11 },\n      xaxis: { gridcolor: \'#21262d\', tickfont:{color:\'#8b949e\'} },\n      yaxis: { gridcolor: \'#21262d\', tickfont:{color:\'#8b949e\'} },\n      legend:{ font:{color:\'#8b949e\'} },\n      margin:{ t:10, b:40, l:50, r:10 }\n    };\n  }\n\n  // Auto load on start\n  window.onload = () => analyzeStock();\n</script>\n</body>\n</html>\n',
}

for name, content in files.items():
    path = base / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")

print("✅ Project files created in:", base)
print("Files:")
for p in sorted(base.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(base))


✅ Project files created in: /content/stock_predictor
Files:
 - README.md
 - app.py
 - lstm_model.py
 - requirements.txt
 - sentiment.py
 - templates/index.html


## 3. Colab Compatibility

In [3]:

# Small Colab compatibility patch:
# 1) Uses flexible package versions instead of old pinned ones.
# 2) Keeps Flask server debug/reloader off in Colab.
print("✅ Compatibility ready. Run the next cell to start the app.")


✅ Compatibility ready. Run the next cell to start the app.


## 4. Quick Test Stock Data

Run this to confirm Yahoo Finance data is loading.

In [4]:

%cd /content/stock_predictor

from app import get_stock_data
ticker = "AAPL"
df, info = get_stock_data(ticker, period="1mo")

print("Ticker:", ticker)
print("Company:", info.get("longName", ticker))
display(df.tail())


/content/stock_predictor
Ticker: AAPL
Company: Apple Inc.


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-06-12 00:00:00-04:00,296.029999,297.140015,289.619995,291.130005,38742100,0.0,0.0
2026-06-15 00:00:00-04:00,294.119995,297.779999,291.700012,296.420013,45732600,0.0,0.0
2026-06-16 00:00:00-04:00,295.250000,300.480011,293.970001,299.239990,39874400,0.0,0.0
2026-06-17 00:00:00-04:00,300.850006,302.070007,294.359985,295.950012,42745100,0.0,0.0
2026-06-18 00:00:00-04:00,298.109985,300.570007,295.619995,298.010010,85962200,0.0,0.0


## 5. Run the Flask Dashboard in Colab

After this cell starts, Colab will open the dashboard.

In [5]:

%cd /content/stock_predictor

import threading, time
try:
    from google.colab import output
except Exception:
    output = None

from app import app

def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()

time.sleep(3)

if output:
    print("✅ Flask app is running. Opening inside Colab...")
    output.serve_kernel_port_as_window(5000)
else:
    print("Open this in browser: http://127.0.0.1:5000")


/content/stock_predictor
 * Serving Flask app 'app'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


✅ Flask app is running. Opening inside Colab...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## 6. Optional: Train LSTM Model

Use this after the dashboard works. Training can take a few minutes.

In [6]:

%cd /content/stock_predictor

# Optional: Train LSTM model.
# It may take a few minutes in Colab.
from lstm_model import train_and_save

ticker = "AAPL"   # Change to MSFT, TSLA, GOOGL, AMZN, etc.
model, scaler, df = train_and_save(ticker)

print("✅ Training completed for:", ticker)


/content/stock_predictor
📈 AAPL ka data download ho raha hai...


/content/stock_predictor/lstm_model.py:18: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period=period, progress=False)


✅ 501 days ka data mila!
🧠 LSTM model train ho raha hai...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 104ms/step - loss: 0.0690 - val_loss: 0.0257
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - loss: 0.0154 - val_loss: 0.0361
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - loss: 0.0098 - val_loss: 0.0037
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - loss: 0.0086 - val_loss: 0.0070
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - loss: 0.0073 - val_loss: 0.0049
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - loss: 0.0063 - val_loss: 0.0039
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - loss: 0.0057 - val_loss: 0.0038
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - loss: 0.0052 - val_loss: 0.0038


✅ Model saved: models/AAPL_lstm.h5
✅ Training completed for: AAPL
